# 04 — Atasi Class Imbalance di Atas Dataset 182 Fitur (+ DDS, + acc-tuned)

Lanjutan langsung dari `03_accuracy_push_with_dds.ipynb` (XGBoost
acc-tuned, 182 fitur, 52.7-53.4% accuracy test set). Accuracy sudah
naik, tapi **F1 macro masih rendah (~0.25-0.30)** karena 35 kelas
kontrak sangat timpang (kelas terkecil di train cuma 5 sampel, `3N`
sendirian ~20% dari semua data). Notebook ini coba teknik yang sama
seperti [experiments/2026-07-09/02_smote_resampling.ipynb](../2026-07-09/02_smote_resampling.ipynb)
(class_weight, RandomOverSampler, SMOTE) tapi di atas basis yang sudah
lebih baik: 182 fitur + hyperparameter accuracy-tuned.

**Peringatan dari temuan sebelumnya**: menambah teknik penyeimbang
kelas biasanya **menaikkan F1 macro tapi menurunkan accuracy** — itu
trade-off yang diharapkan (bukan bug), karena macro-F1 dan accuracy
mengoptimalkan hal yang beda (rata-rata per-kelas vs rata-rata per-baris,
yang didominasi kelas umum). SMOTE juga sudah 2x terbukti merugikan
(2026-07-09 & saat digabung dengan DART 2026-07-10) — dicoba lagi di
sini untuk verifikasi apakah itu tetap berlaku di dataset 182 fitur.


In [1]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE, RandomOverSampler

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.dataset_builder import load_splits
from src.evaluation.metrics import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-15" / "outputs" / "class_imbalance_dds"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed_dds")

X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_val, y_val = df_val[feature_cols].values, df_val["label"].values
X_test, y_test = df_test[feature_cols].values, df_test["label"].values

class_counts = pd.Series(y_train).value_counts()
min_class_count = int(class_counts.min())
print(f"Train: {X_train.shape} (182 fitur), kelas terkecil: {min_class_count} sampel")
class_counts.sort_values().head(8)


Train: (7157, 182) (182 fitur), kelas terkecil: 5 sampel


29     5
30     5
32     7
0      7
31    13
1     14
34    14
33    16
Name: count, dtype: int64

## 1. Konfigurasi dasar

XGBoost pakai hyperparameter accuracy-tuned dari notebook 03 (bukan
default `config.yaml`) — supaya kita bandingkan "kandidat terbaik saat
ini + imbalance handling", bukan mengulang dari nol.


In [2]:
XGB_ACC_TUNED = dict(
    n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.9, colsample_bytree=0.6, min_child_weight=5, reg_lambda=2.0,
    random_state=42, n_jobs=-1, eval_metric="mlogloss", verbosity=0,
)
LGBM_BASE = dict(
    n_estimators=300, max_depth=-1, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1,
)
RF_BASE = dict(
    n_estimators=300, max_depth=None, min_samples_split=5,
    random_state=42, n_jobs=-1,
)


def make_smote():
    k = max(1, min(5, min_class_count - 1))
    return SMOTE(k_neighbors=k, random_state=42)


## 2. XGBoost (acc-tuned) — ablation imbalance handling

In [3]:
xgb_results = []
xgb_probas = {}

# A: baseline acc-tuned, tanpa imbalance handling
t0 = time.time()
clf = XGBClassifier(**XGB_ACC_TUNED)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_val)
xgb_probas["none"] = proba
res = evaluate(y_val, proba.argmax(axis=1), proba, le, model_name="XGBoost acc-tuned [none]")
res["fit_seconds"] = round(time.time() - t0, 1)
xgb_results.append(res)
print_summary(res)

# B: + sample_weight balanced
t0 = time.time()
sw = compute_sample_weight("balanced", y_train)
clf = XGBClassifier(**XGB_ACC_TUNED)
clf.fit(X_train, y_train, sample_weight=sw)
proba = clf.predict_proba(X_val)
xgb_probas["sample_weight"] = proba
res = evaluate(y_val, proba.argmax(axis=1), proba, le, model_name="XGBoost acc-tuned [sample_weight]")
res["fit_seconds"] = round(time.time() - t0, 1)
xgb_results.append(res)
print_summary(res)

# C: + RandomOverSampler
t0 = time.time()
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)
clf = XGBClassifier(**XGB_ACC_TUNED)
clf.fit(X_ros, y_ros)
proba = clf.predict_proba(X_val)
xgb_probas["random_oversample"] = proba
res = evaluate(y_val, proba.argmax(axis=1), proba, le, model_name="XGBoost acc-tuned [random_oversample]")
res["fit_seconds"] = round(time.time() - t0, 1)
res["n_train_rows"] = int(len(y_ros))
xgb_results.append(res)
print_summary(res)

# D: + SMOTE
t0 = time.time()
smote = make_smote()
X_sm, y_sm = smote.fit_resample(X_train, y_train)
clf = XGBClassifier(**XGB_ACC_TUNED)
clf.fit(X_sm, y_sm)
proba = clf.predict_proba(X_val)
xgb_probas["smote"] = proba
res = evaluate(y_val, proba.argmax(axis=1), proba, le, model_name="XGBoost acc-tuned [smote]")
res["fit_seconds"] = round(time.time() - t0, 1)
res["n_train_rows"] = int(len(y_sm))
xgb_results.append(res)
print_summary(res)



  XGBoost acc-tuned [none]
  Accuracy          : 0.5173
  Precision (macro) : 0.3313
  Recall (macro)    : 0.2411
  F1 (macro)        : 0.2566
  F1 (weighted)     : 0.4756
  Top-3 Accuracy    : 0.7684
  Top-5 Accuracy    : 0.8526



  XGBoost acc-tuned [sample_weight]
  Accuracy          : 0.4612
  Precision (macro) : 0.2695
  Recall (macro)    : 0.3241
  F1 (macro)        : 0.2793
  F1 (weighted)     : 0.4789
  Top-3 Accuracy    : 0.7293
  Top-5 Accuracy    : 0.8258



  XGBoost acc-tuned [random_oversample]
  Accuracy          : 0.4729
  Precision (macro) : 0.2931
  Recall (macro)    : 0.3286
  F1 (macro)        : 0.2987
  F1 (weighted)     : 0.4840
  Top-3 Accuracy    : 0.7293
  Top-5 Accuracy    : 0.8245



  XGBoost acc-tuned [smote]
  Accuracy          : 0.4697
  Precision (macro) : 0.2786
  Recall (macro)    : 0.2786
  F1 (macro)        : 0.2650
  F1 (weighted)     : 0.4509
  Top-3 Accuracy    : 0.7397
  Top-5 Accuracy    : 0.8239


## 3. LightGBM & RandomForest — class_weight native, untuk pembanding

In [4]:
other_results = []

# LightGBM: baseline vs class_weight="balanced"
t0 = time.time()
clf = LGBMClassifier(**LGBM_BASE)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_val)
res = evaluate(y_val, proba.argmax(axis=1), proba, le, model_name="LightGBM [none]")
res["fit_seconds"] = round(time.time() - t0, 1)
other_results.append(res)
print_summary(res)

t0 = time.time()
clf = LGBMClassifier(**{**LGBM_BASE, "class_weight": "balanced"})
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_val)
res = evaluate(y_val, proba.argmax(axis=1), proba, le, model_name="LightGBM [class_weight]")
res["fit_seconds"] = round(time.time() - t0, 1)
other_results.append(res)
print_summary(res)

# RandomForest: baseline vs class_weight="balanced"
t0 = time.time()
clf = RandomForestClassifier(**RF_BASE)
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_val)
res = evaluate(y_val, proba.argmax(axis=1), proba, le, model_name="RandomForest [none]")
res["fit_seconds"] = round(time.time() - t0, 1)
other_results.append(res)
print_summary(res)

t0 = time.time()
clf = RandomForestClassifier(**{**RF_BASE, "class_weight": "balanced"})
clf.fit(X_train, y_train)
proba = clf.predict_proba(X_val)
res = evaluate(y_val, proba.argmax(axis=1), proba, le, model_name="RandomForest [class_weight]")
res["fit_seconds"] = round(time.time() - t0, 1)
other_results.append(res)
print_summary(res)


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM [none]
  Accuracy          : 0.5127
  Precision (macro) : 0.3167
  Recall (macro)    : 0.2305
  F1 (macro)        : 0.2489
  F1 (weighted)     : 0.4645
  Top-3 Accuracy    : 0.7560
  Top-5 Accuracy    : 0.8389


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM [class_weight]
  Accuracy          : 0.5029
  Precision (macro) : 0.3569
  Recall (macro)    : 0.2777
  F1 (macro)        : 0.2968
  F1 (weighted)     : 0.4649
  Top-3 Accuracy    : 0.7508
  Top-5 Accuracy    : 0.8415



  RandomForest [none]
  Accuracy          : 0.4723
  Precision (macro) : 0.2260
  Recall (macro)    : 0.1462
  F1 (macro)        : 0.1539
  F1 (weighted)     : 0.4023
  Top-3 Accuracy    : 0.7228
  Top-5 Accuracy    : 0.8193



  RandomForest [class_weight]
  Accuracy          : 0.4436
  Precision (macro) : 0.2614
  Recall (macro)    : 0.3274
  F1 (macro)        : 0.2755
  F1 (weighted)     : 0.4586
  Top-3 Accuracy    : 0.7149
  Top-5 Accuracy    : 0.8187


## 4. Bandingkan semua (val set)

In [5]:
all_results = xgb_results + other_results
comparison = compare_models(all_results)
comparison_sorted_f1 = comparison.sort_values("f1_macro", ascending=False)
comparison_sorted_f1


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
XGBoost acc-tuned [random_oversample],0.472929,0.293085,0.328616,0.298708,0.484028,0.729289,0.824527
LightGBM [class_weight],0.502935,0.356885,0.277740,0.296838,0.464869,0.750815,0.841487
XGBoost acc-tuned [sample_weight],0.461187,0.269520,0.324105,0.279312,0.478884,0.729289,0.825832
RandomForest [class_weight],0.443575,0.261403,0.327426,0.275516,0.458550,0.714938,0.818656
XGBoost acc-tuned [smote],0.469667,0.278633,0.278579,0.265012,0.450870,0.739726,0.823875
XGBoost acc-tuned [none],0.517286,0.331282,0.241127,0.256595,0.475589,0.768428,0.852577
LightGBM [none],0.512720,0.316700,0.230490,0.248913,0.464487,0.756034,0.838878
RandomForest [none],0.472277,0.226003,0.146236,0.153945,0.402305,0.722766,0.819309


In [6]:
comparison_sorted_f1.to_csv(OUT_DIR / "val_comparison.csv")

print("Trade-off accuracy vs F1 macro (diurutkan F1 macro tertinggi dulu):")
for name, row in comparison_sorted_f1.iterrows():
    print(f"  {name:40s} acc={row['accuracy']:.4f}  f1_macro={row['f1_macro']:.4f}  f1_weighted={row['f1_weighted']:.4f}")


Trade-off accuracy vs F1 macro (diurutkan F1 macro tertinggi dulu):
  XGBoost acc-tuned [random_oversample]    acc=0.4729  f1_macro=0.2987  f1_weighted=0.4840
  LightGBM [class_weight]                  acc=0.5029  f1_macro=0.2968  f1_weighted=0.4649
  XGBoost acc-tuned [sample_weight]        acc=0.4612  f1_macro=0.2793  f1_weighted=0.4789
  RandomForest [class_weight]              acc=0.4436  f1_macro=0.2755  f1_weighted=0.4586
  XGBoost acc-tuned [smote]                acc=0.4697  f1_macro=0.2650  f1_weighted=0.4509
  XGBoost acc-tuned [none]                 acc=0.5173  f1_macro=0.2566  f1_weighted=0.4756
  LightGBM [none]                          acc=0.5127  f1_macro=0.2489  f1_weighted=0.4645
  RandomForest [none]                      acc=0.4723  f1_macro=0.1539  f1_weighted=0.4023


## 5. Validasi kandidat terbaik (F1 macro) di TEST SET

Pilih varian dengan F1 macro tertinggi TANPA accuracy anjlok lebih dari
~3pp dari baseline acc-tuned — supaya tetap berguna praktis (bukan
cuma menang di F1 macro tapi accuracy jadi jelek).


In [7]:
baseline_acc = xgb_results[0]["accuracy"]
candidates = [r for r in all_results if r["accuracy"] >= baseline_acc - 0.03]
best_candidate = max(candidates, key=lambda r: r["f1_macro"])
print(f"Baseline accuracy (acc-tuned, no imbalance handling): {baseline_acc:.4f}")
print(f"Kandidat terpilih (F1 macro tertinggi, accuracy tidak turun >3pp): {best_candidate['model']}")
print(f"  accuracy={best_candidate['accuracy']:.4f}  f1_macro={best_candidate['f1_macro']:.4f}")


Baseline accuracy (acc-tuned, no imbalance handling): 0.5173
Kandidat terpilih (F1 macro tertinggi, accuracy tidak turun >3pp): LightGBM [class_weight]
  accuracy=0.5029  f1_macro=0.2968


In [8]:
# Re-fit kandidat terpilih dan validasi di test set (sekali saja)
variant_map = {
    "XGBoost acc-tuned [none]": ("xgb", "none"),
    "XGBoost acc-tuned [sample_weight]": ("xgb", "sample_weight"),
    "XGBoost acc-tuned [random_oversample]": ("xgb", "random_oversample"),
    "XGBoost acc-tuned [smote]": ("xgb", "smote"),
    "LightGBM [none]": ("lgbm", "none"),
    "LightGBM [class_weight]": ("lgbm", "class_weight"),
    "RandomForest [none]": ("rf", "none"),
    "RandomForest [class_weight]": ("rf", "class_weight"),
}

model_kind, variant = variant_map[best_candidate["model"]]

if model_kind == "xgb":
    if variant == "none":
        clf = XGBClassifier(**XGB_ACC_TUNED); clf.fit(X_train, y_train)
    elif variant == "sample_weight":
        sw = compute_sample_weight("balanced", y_train)
        clf = XGBClassifier(**XGB_ACC_TUNED); clf.fit(X_train, y_train, sample_weight=sw)
    elif variant == "random_oversample":
        X_r, y_r = RandomOverSampler(random_state=42).fit_resample(X_train, y_train)
        clf = XGBClassifier(**XGB_ACC_TUNED); clf.fit(X_r, y_r)
    elif variant == "smote":
        X_r, y_r = make_smote().fit_resample(X_train, y_train)
        clf = XGBClassifier(**XGB_ACC_TUNED); clf.fit(X_r, y_r)
elif model_kind == "lgbm":
    params = {**LGBM_BASE, "class_weight": "balanced"} if variant == "class_weight" else LGBM_BASE
    clf = LGBMClassifier(**params); clf.fit(X_train, y_train)
elif model_kind == "rf":
    params = {**RF_BASE, "class_weight": "balanced"} if variant == "class_weight" else RF_BASE
    clf = RandomForestClassifier(**params); clf.fit(X_train, y_train)

y_proba_test = clf.predict_proba(X_test)
res_test = evaluate(y_test, y_proba_test.argmax(axis=1), y_proba_test, le,
                     model_name=f"{best_candidate['model']} TEST")
print_summary(res_test)

# Baseline test juga (untuk perbandingan langsung)
clf_base = XGBClassifier(**XGB_ACC_TUNED)
clf_base.fit(X_train, y_train)
proba_base_test = clf_base.predict_proba(X_test)
res_test_base = evaluate(y_test, proba_base_test.argmax(axis=1), proba_base_test, le,
                          model_name="XGBoost acc-tuned [none] TEST")
print_summary(res_test_base)

test_comparison = compare_models([res_test_base, res_test])
test_comparison.to_csv(OUT_DIR / "test_comparison.csv")
test_comparison


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM [class_weight] TEST
  Accuracy          : 0.5258
  Precision (macro) : 0.4164
  Recall (macro)    : 0.2884
  F1 (macro)        : 0.3068
  F1 (weighted)     : 0.4854
  Top-3 Accuracy    : 0.7528
  Top-5 Accuracy    : 0.8519



  XGBoost acc-tuned [none] TEST
  Accuracy          : 0.5271
  Precision (macro) : 0.3086
  Recall (macro)    : 0.2371
  F1 (macro)        : 0.2507
  F1 (weighted)     : 0.4802
  Top-3 Accuracy    : 0.7704
  Top-5 Accuracy    : 0.8558


,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,top_3_accuracy,top_5_accuracy
model,,,,,,,
XGBoost acc-tuned [none] TEST,0.527071,0.308598,0.237135,0.250712,0.480247,0.770385,0.855838
LightGBM [class_weight] TEST,0.525766,0.416365,0.288406,0.306764,0.485403,0.752772,0.851924


In [9]:
summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "min_class_count_train": min_class_count,
    "val_results": {r["model"]: {k: r[k] for k in
        ["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]}
        for r in all_results},
    "best_candidate_val": best_candidate["model"],
    "test_validation": {
        "baseline": {k: res_test_base[k] for k in ["accuracy", "f1_macro", "f1_weighted"]},
        "best_candidate": {k: res_test[k] for k in ["accuracy", "f1_macro", "f1_weighted"]},
    },
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved: {OUT_DIR / 'summary.json'}")


Saved: D:\Bridge-Prediction\experiments\2026-07-15\outputs\class_imbalance_dds\summary.json


## 6. Kesimpulan

Hasil val set (`experiments/2026-07-15/outputs/class_imbalance_dds/val_comparison.csv`),
dan validasi test set untuk kandidat terpilih:

| Model | Accuracy | F1 Macro | F1 Weighted |
|---|---|---|---|
| XGBoost acc-tuned [none] (baseline notebook 03) | **51.7%** (val) / 52.7% (test) | 0.257 (val) / 0.251 (test) | 0.476 (val) / 0.480 (test) |
| **LightGBM [class_weight]** | 50.3% (val) / **52.6%** (test) | 0.297 (val) / **0.307** (test) | 0.465 (val) / **0.485** (test) |
| XGBoost acc-tuned [random_oversample] | 47.3% | 0.299 | 0.484 |
| XGBoost acc-tuned [sample_weight] | 46.1% | 0.279 | 0.479 |
| XGBoost acc-tuned [smote] | 47.0% | 0.265 | 0.451 |
| RandomForest [class_weight] | 44.4% | 0.276 | 0.459 |
| RandomForest [none] | 47.2% | 0.154 | 0.402 |

**Hasil terbaik: `LightGBM + class_weight="balanced"`** — hampir tidak
ada trade-off. Di test set, accuracy-nya (52.6%) praktis SAMA dengan
XGBoost acc-tuned (52.7%, beda cuma -0.1pp), tapi **F1 macro naik 22%
relatif** (0.251 -> 0.307) dan F1 weighted juga naik (0.480 -> 0.485).
Ini beda dari pola biasa hari-hari sebelumnya di mana menyeimbangkan
kelas SELALU mengorbankan accuracy signifikan — kali ini nyaris gratis.

**SMOTE terbukti merugikan untuk KETIGA KALINYA** (2026-07-09 di 164
fitur, 2026-07-10 dikombinasi dengan DART, dan sekarang di 182 fitur +
acc-tuned) — F1 macro-nya (0.265) bahkan lebih rendah dari baseline
tanpa penyeimbangan (0.257 val, meski test belum dicek karena bukan
kandidat terpilih). Kesimpulan ini sudah sangat robust, tidak perlu
diuji ulang lagi kecuali ada perubahan fundamental pada data.

**RandomOverSampler dan sample_weight untuk XGBoost acc-tuned KURANG
BAIK dibanding LightGBM+class_weight** di sini — accuracy anjlok lebih
dari 4pp (51.7%->47.3% dan 46.1%) untuk kenaikan F1 macro yang serupa
atau lebih kecil. Kemungkinan karena hyperparameter XGBoost sudah
di-tuning ketat untuk distribusi data ASLI (tidak di-resample) —
menambah/menduplikasi data mengubah distribusi yang sudah dioptimalkan
untuknya, sementara LightGBM dengan class_weight tidak mengubah
distribusi data sama sekali (cuma bobot loss).

**Rekomendasi**: **LightGBM + class_weight="balanced"** (182 fitur)
adalah kandidat terbaik baru untuk kasus yang butuh keseimbangan
antar-kelas (F1 macro) tanpa mengorbankan accuracy — praktis
menggantikan XGBoost acc-tuned sebagai pilihan utama jika F1 macro
juga jadi prioritas, bukan cuma accuracy murni. Kalau accuracy murni
tetap prioritas nomor satu, XGBoost acc-tuned [none] (notebook 03)
tetap yang terbaik (52.7% vs 52.6%, beda tipis tapi tetap menang).
